# 2.2 · LGD fine-tuning

*2. Experiment 2 · notebook 2.2 of the story.* ← [2.1 · PD fine-tuning](<2.1_pd_finetuning.ipynb>) · [2.3 · PD results](<2.3_pd_results.ipynb>) →

**Experiment 2 asks: is the credit prior useful for the real, fully trained foundation
models?** It continues the pretraining of the released TabICLv2 weights (`checkpoints/tabicl-regressor-v2-20260212.ckpt`) and
of the released TabPFN-3 weights (`checkpoints/tabpfn-v3-regressor-v3_default.ckpt`) on Experiment 1's prior for 10,000 steps —
64 tasks per step for TabICLv2, 16 for TabPFN-3, as in TabPFN-Wide's continued pretraining — at three
prior mixes with 3 seeds each: 2 models × 3 mixes × 3 seeds = 18 arms. The recipe — optimizer,
learning rate and length — is chosen first, per model, by a search on the development datasets with
the control mix (`config/Exp2_LGD_search.yaml`, read by `python -m src.eval.exp2_search`), so it is
not tuned in favour of our prior; `config/Exp2_LGD.yaml` holds `FILL_FROM_SEARCH` until then.

**How the arms are named.** In every figure an arm reads `TabPFN-3 · 50 % credit · seed 0`: the
released model whose pretraining continues, the prior mix (`0 % credit` is continued pretraining on
TabICL's prior alone — the control) and the seed. The reference models are the `released TabICLv2`,
`TabPFN-3`, `CatBoost` and `linear regression`; for this experiment the two released models are the
starting points every arm is measured against.

**Two things a reader must know.** TabPFN's own prior is not public, so the TabPFN-3 arms continue on
TabICL's prior (the control) and ours; every arm shares that, so the effect of our prior is still
isolated. And every arm trains on 1,024-row tables, TabICLv2's first-stage size, where both released
models also saw far longer ones; the control shares that too, so the comparison between mixes is
clean, while the comparison with the released weights mixes our prior's effect with that change.

**Status: no Experiment 2 arm has run.** The sweep map (A1) shows the 18 scheduled arms; every other
figure is a one-line placeholder until arms have run, and fills in as they do. The structure — and the
three averaging rules — are Experiment 1's ([1.1 · PD training](<../1. Experiment 1/1.1_pd_training.ipynb>)).

**The order.** **A** — is the run sound: which arms exist and finished, what each was scored on,
what each cost, whether each learned. **B** — what the development monitoring says about each lever.
**C** — whether it holds on every dataset, outside credit and on every metric. **D** — every arm on
its own.

In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys, pathlib
ROOT = pathlib.Path.cwd()
# Walk up to the repository root — the notebook may be opened from its chapter folder, from
# notebooks/, or from the root — then work FROM the root, so relative paths (config/...) resolve
# exactly as under `python -m src.utils.run_notebooks`.
while not (ROOT / "src" / "visualize").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from src.visualize import training_plots, figures, style, literature

style.apply()
pd.set_option("display.width", 200, "display.max_columns", 40)

TASK = "lgd"
EXP = "exp2"
# Reads output CreditICL/experiment_2/ (runs/, benchmark/) — whatever the runs have written so far, so a
# partial sweep still renders. Constructing the saver clears THIS notebook's figure folder only.
FIGS = figures.FigureSaver("2.2_lgd_finetuning")

## A · Is the run sound?

Before any number is read: which arms exist and finished, what each was scored on, what each cost,
and whether each one learned.

### A1 · Which arms exist, and how did each finish?

**What it shows.** Every scheduled arm in one grid: one row per configuration, grouped by credit fraction, one column per seed (for a one-seed sweep, one column per credit fraction instead). A number is the arm's final development R² — the mean over the development datasets every arm scored; a grey cell has no development score; a hatched cell is still training and says how far it has got.

**Why it matters.** A reader has to know which arms exist, which finished, and whether a block of rows or one seed stands out, before any comparison means anything: an effect shared by one seed moves every comparison that is not averaged over seeds.

**What it says.** Written from the redesigned run once its results are downloaded; until then the figure above is a placeholder. (The reading of the September run, which used the retired design, is in the git history.)

In [ ]:
FIGS.save(training_plots.sweep_map(TASK, exp=EXP), "sweep_map",
    caption="Final development R² of every scheduled LGD arm as a heatmap, one row per configuration grouped by prior mix and one column per seed; grey cells have no development score, hatched cells are still training with the share of steps done.");

### A2 · What was each arm scored on?

**What it shows.** One row per real dataset (development first, then holdout), one column per arm in the sweep map's order. A cell's colour says what that arm's score on that dataset may do: enter a development mean, exist without being averaged (not every arm carries it), be a holdout score (never averaged), or be a monitor that logged only missing values.

**Why it matters.** Arms were monitored under two protocols. Before 23-09-2026 the monitor scored "the four smallest" real datasets whatever their role, holdout ones included; since then, development datasets only. Averaging a holdout score into "which prior is best" would select on the test set, so this map fixes exactly which cells every curve in Part B is built from.

**What it says.** Written from the redesigned run once its results are downloaded; until then the figure above is a placeholder. (The reading of the September run, which used the retired design, is in the git history.)

In [ ]:
FIGS.save(training_plots.monitoring_coverage(TASK, exp=EXP), "monitoring_coverage",
    caption="Monitoring coverage of every LGD arm: one row per real dataset labelled development or holdout, one column per arm grouped by prior mix, each cell coloured by whether its score enters the development mean, is not carried by every arm, is holdout, holds only missing values or was not monitored.");

### A3 · What did each arm cost?

**What it shows.** Each arm's training speed — its median steps per second over the run — grouped by the released model continued, and the hours the full run takes at that speed; the bar is the group median, points are coloured and shaped by credit fraction.

**Why it matters.** TabPFN-3 preprocesses every task on the CPU, in the data-loader workers, before its GPU step, and TabICLv2 does not — so the two models' speeds differ for a reason that has nothing to do with the prior, and each model's cost in walltime and VSC credits has to be read on its own.

**What it says.** Written from the redesigned run once its results are downloaded; until then the figure above is a placeholder. (The reading of the September run, which used the retired design, is in the git history.)

In [ ]:
FIGS.save(training_plots.throughput(TASK, exp=EXP), "throughput",
    caption="Training speed per LGD arm in steps per second (left) and the implied hours for the full run (right), grouped by layers trained; points are arms coloured and shaped by prior mix and bars are group medians with their value.");

### A4 · Did every arm learn?

**What it shows.** The training loss of every arm against the step — the pinball loss over 999 quantiles on the arm's own synthetic batch, as a rolling mean over three logged points (each logged value is one batch's loss) — with the median of the finished arms of each credit fraction drawn bold.

**Why it matters.** The loss is the one quantity every arm optimises directly, so divergence or a stall shows here first; TabICLv2's ablations warn that a prior can make training diverge (`papers/2026/02_Qu_TabICLv2` §4.4). But it is computed on each arm's own prior, so its level says which prior an arm trains on — not how well it learned.

**What it says.** Written from the redesigned run once its results are downloaded; until then the figure above is a placeholder. (The reading of the September run, which used the retired design, is in the git history.)

In [ ]:
FIGS.save(training_plots.training_loss(TASK, exp=EXP), "training_loss",
    caption="Training loss against optimisation step for every LGD arm, on a logarithmic axis, a rolling mean over five values logged every 100 steps, coloured by prior mix and dotted while unfinished, with the median of the finished arms of each prior mix bold.");

### A5 · Is every part of the network learning?

**What it shows.** Left, each block's gradient norm; right, the gradient divided by the block's weight norm — how far one step moves the block relative to its size. Both are averaged over all arms, on logarithmic axes. The telemetry records the column encoder, the row encoder and the ICL blocks; the prediction head is not logged.

**Why it matters.** A block whose gradient sits orders of magnitude below the others is effectively frozen, and the loss cannot say which one. In continued pretraining every block trains, so a block on the floor would be one the optimizer failed to move.

**What it says.** Written from the redesigned run once its results are downloaded; until then the figure above is a placeholder. (The reading of the September run, which used the retired design, is in the git history.)

In [ ]:
FIGS.save(training_plots.gradient_health(TASK, exp=EXP), "gradient_health",
    caption="Mean per-block gradient L2 norm (left) and ratio of gradient norm to weight norm (right) against training step on logarithmic axes, one line per architecture block (column encoder, row encoder, ICL blocks), averaged over all arms.");

## B · What does the development monitoring say?

The development R² over training and at its end, lever by lever. It guides the eye; the benchmark in
[2.4 · LGD results](<2.4_lgd_results.ipynb>) decides.

### B1 · Development R² over training

**What it shows.** The development R² of every arm against the step, averaged over the development datasets every arm scored, in its credit-fraction colour and dotted while unfinished, with the mean of the finished arms of each fraction drawn bold.

**Why it matters.** The first look at how models trained on each prior do on real credit data they never trained on — and whether the fractions separate early, late or not at all.

**What it says.** Written from the redesigned run once its results are downloaded; until then the figure above is a placeholder. (The reading of the September run, which used the retired design, is in the git history.)

In [ ]:
FIGS.save(training_plots.metric_over_training(TASK, exp=EXP), "metric_over_training",
    caption="Development R² averaged over the development datasets every arm scored, against training step; one line per arm coloured by prior mix and dotted while unfinished, with the mean of the finished arms of each prior mix bold.");

### B2 · Credit prior versus the control

**What it shows.** Credit-prior arms (any credit fraction above 0) against the control arms (credit fraction 0 — TabICL's own prior): the median over the finished arms of each group, with the inter-quartile band.

**Why it matters.** The experiment's comparison, as a curve. A gap narrower than the bands is not read as an effect.

**What it says.** Written from the redesigned run once its results are downloaded; until then the figure above is a placeholder. (The reading of the September run, which used the retired design, is in the git history.)

In [ ]:
FIGS.save(training_plots.credit_vs_control_over_training(TASK, exp=EXP), "credit_vs_control_over_training",
    caption="Development R² against training step for arms with our credit prior in the mix and for control arms: median lines with shaded inter-quartile bands over the finished arms of each group.");

### B3 · Which lever moves the score?

**What it shows.** One panel per swept lever (the prior mix and the released model continued), one mean curve per value over the finished arms that share it, all panels on one y axis.

**Why it matters.** The size of each lever's effect, side by side: a lever whose curves lie on top of each other does nothing, whatever the others do.

**What it says.** Written from the redesigned run once its results are downloaded; until then the figure above is a placeholder. (The reading of the September run, which used the retired design, is in the git history.)

In [ ]:
FIGS.save(training_plots.metric_by_levers(TASK, exp=EXP), "metric_by_levers",
    caption="Development R² against training step, one panel per swept lever with one mean line per lever value over the finished arms sharing it, all panels on a shared vertical axis.");

### B4 · Final score, lever by lever

**What it shows.** Every finished arm's final development score, grouped by each lever in turn: points coloured and shaped by credit fraction, a bar at each group mean, one shared y axis.

**Why it matters.** Final scores as a screen, one lever at a time. The sweep is factorial, so a lever's groups can differ merely because they contain different credit fractions; B5 separates the two.

**What it says.** Written from the redesigned run once its results are downloaded; until then the figure above is a placeholder. (The reading of the September run, which used the retired design, is in the git history.)

In [ ]:
FIGS.save(training_plots.final_metric_by_lever(TASK, exp=EXP), "final_metric_by_lever",
    caption="Final development R² of every finished arm as points coloured and shaped by prior mix, one panel per swept lever, with a horizontal bar at each group mean.");

### B5 · Each lever within each credit fraction

**What it shows.** Each lever other than the credit fraction, drawn within each fraction: points are finished arms, lines join the per-value means, whiskers are ±1 standard deviation over seeds. A lever the control does not have appears as the control's band (its mean ± 1 SD) instead.

**Why it matters.** Pooling a lever over fractions mixes the fraction's larger effect into it; within one fraction, a real effect of the other levers has nowhere to hide.

**What it says.** Written from the redesigned run once its results are downloaded; until then the figure above is a placeholder. (The reading of the September run, which used the retired design, is in the git history.)

In [ ]:
FIGS.save(training_plots.lever_interaction(TASK, exp=EXP), "lever_interaction",
    caption="Final development R² against each lever other than the prior mix, one series per prior mix: points are finished arms, lines join the per-value means, whiskers show one standard deviation over seeds, and the grey band is the control mean plus or minus one standard deviation.");

### B6 · Effect against seed noise

**What it shows.** Every configuration's seeds on one axis, best configuration at the top: points are seeds, the line their range, the tick their mean; the dashed line is the control's mean.

**Why it matters.** A difference between two configurations is only worth reading where it clearly exceeds the spread between one configuration's own seeds — Experiment 2 runs one seed, so this figure will show single runs.

**What it says.** Written from the redesigned run once its results are downloaded; until then the figure above is a placeholder. (The reading of the September run, which used the retired design, is in the git history.)

In [ ]:
FIGS.save(training_plots.seed_spread(TASK, exp=EXP), "seed_spread",
    caption="Final development R² of every configuration's seeds as points on one horizontal axis, configurations sorted by their mean with the range and mean marked per row; the dashed line is the control mean.");

## C · Where does it hold?

A mean can be carried by one easy dataset, and a gain on credit can be bought with general ability.
These figures check both, and every metric the monitor records.

### C1 · Every monitored dataset

**What it shows.** One panel per real dataset the monitor scored, development first: the mean over the finished credit-prior arms and over the finished control arms that scored it. Each title gives the dataset's side of the split and how many finished arms scored it.

**Why it matters.** Development panels are what the means above are built from. Holdout panels are shown as the evidence they are and never enter a mean — they belong to the benchmark.

**What it says.** Written from the redesigned run once its results are downloaded; until then the figure above is a placeholder. (The reading of the September run, which used the retired design, is in the git history.)

In [ ]:
for _p in range(1, training_plots.per_dataset_pages(TASK, exp=EXP) + 1):
    FIGS.save(training_plots.per_dataset_curves(TASK, _p, exp=EXP), f"per_dataset_p{_p}",
        caption="Monitored R² against training step, one panel per real dataset labelled with its side of the split and the number of finished arms that scored it, arms with our credit prior in the mix against control arms.");

### C2 · Credit data against out-of-domain data

**What it shows.** The development score on the real credit datasets (solid) and on eight public regression suites from outside credit (dashed), each averaged separately over credit-prior and control arms.

**Why it matters.** Out-of-domain retention is a first-class axis of the design (`docs/EXPERIMENTAL_DESIGN.md` §5.3): a prior that lifts credit scores by making the model worse everywhere else has not made it better.

**What it says.** Written from the redesigned run once its results are downloaded; until then the figure above is a placeholder. (The reading of the September run, which used the retired design, is in the git history.)

In [ ]:
FIGS.save(training_plots.real_vs_ood(TASK, exp=EXP), "real_vs_ood",
    caption="Development R² on the real-credit datasets (solid) and on the out-of-domain suites (dashed) against training step, averaged separately over arms with our credit prior in the mix and control arms.");

### C3 · Every development metric

**What it shows.** Every metric the monitor records, one figure per group (for LGD: accuracy; ranking and calibration; intervals and boundary atoms), one panel per metric: the mean over finished credit-prior and control arms. A title carries the metric's goal — ↑ higher is better, ↓ lower is better, → a value it should equal, drawn dashed; a predicted boundary share is drawn against the true share in the data.

**Why it matters.** A prior that improves ranking but hurts calibration, or fits the interior but misses the atoms, shows here and nowhere in R².

**What it says.** Written from the redesigned run once its results are downloaded; until then the figure above is a placeholder. (The reading of the September run, which used the retired design, is in the git history.)

In [ ]:
for _p in range(1, training_plots.metric_pages(TASK, exp=EXP) + 1):
    FIGS.save(training_plots.all_eval_metrics(TASK, _p, exp=EXP), f"eval_metrics_p{_p}",
        caption="Each logged development metric against training step, one panel per metric and one figure per metric group, means over the finished arms with our credit prior in the mix and over the control arms; the title marks the improving direction or target value, drawn as a dashed line.");

## D · Every arm on its own

For the reader who wants to check one configuration rather than trust a mean.

### D1 · Every configuration, seed by seed

**What it shows.** One panel per configuration in the sweep map's order: its seeds as separate lines of the development score, all panels on shared axes; a line that stops early belongs to an unfinished arm.

**Why it matters.** An arm that behaved unlike its neighbours, or unlike its own other seeds, stands out here and disappears in every mean.

**What it says.** Written from the redesigned run once its results are downloaded; until then the figure above is a placeholder. (The reading of the September run, which used the retired design, is in the git history.)

In [ ]:
for _p in range(1, training_plots.config_pages(TASK, exp=EXP) + 1):
    FIGS.save(training_plots.per_config(TASK, _p, exp=EXP), f"per_config_p{_p}",
        caption="Development R² against training step, one panel per configuration in sweep-map order with one line per seed, all panels on shared axes; lines that stop early belong to unfinished arms.");

### D2 · The best arm and the worst

**What it shows.** The finished arms with the highest and the lowest final development score: training loss and development score side by side.

**Why it matters.** Did the worst arm fail to learn, or learn something that does not transfer? The loss answers the first question, the score the second.

**What it says.** Written from the redesigned run once its results are downloaded; until then the figure above is a placeholder. (The reading of the September run, which used the retired design, is in the git history.)

In [ ]:
FIGS.save(training_plots.best_and_worst(TASK, exp=EXP), "best_and_worst",
    caption="Training loss (a rolling mean over three logged batches) and development R² against training step for the best and the worst finished arm by final development score.");

## Summary

The sweep in text: the run (A), what the development monitoring says (B) and where it holds (C). Printed last, in the order of the sections above, so `output CreditICL/All_Results.md` carries the
same story as this notebook, followed by the `tfm-library` sources it cites (pin `81c749b`) and the
list of figures.

In [ ]:
print(training_plots.training_summary(TASK, exp=EXP))
print()
print(literature.references_md(["batch", "stage1_lr", "optimizer", "filter_rate_reg", "filter_pval", "filter_extratrees_n"]))
print()
print(FIGS.summary())